In [1]:
# 1. Clonar el repositorio base del taller
!git clone https://github.com/ebuitrago/demo-reg-clas-canciones.git

# 2. Instalar las dependencias con las versiones exactas del taller
!pip install -q -r demo-reg-clas-canciones/requirements.txt

Cloning into 'demo-reg-clas-canciones'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (188/188), done.
remote: Compressing objects: 100% (182/182), done.
remote: Total 188 (delta 78), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (188/188), 183.77 KiB | 3.83 MiB/s, done.
Resolving deltas: 100% (78/78), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.9/443.9 kB 31.5 MB/s eta 0:00:00
   ━━━━

In [1]:
import sys
import pandas as pd
import numpy as np

# Ruta al repositorio clonado
sys.path.append("/content/demo-reg-clas-canciones")

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

from src.variables import COLUMNAS_ENTRADA, derivadas, preprocesamiento
from src.artefacto import guardar

print("✅ Entorno listo y librerías cargadas correctamente.")

✅ Entorno listo y librerías cargadas correctamente.


In [2]:
# Cargar el dataset
df = pd.read_csv("/content/demo-reg-clas-canciones/data/canciones.csv")

# Separar explicativas legítimas (excluye reproducciones_sem1 y es_hit)
X = df[COLUMNAS_ENTRADA]
y = df['popularidad']

# Partición temporal: entrenar <= 2019, probar >= 2020
mask_entrenamiento = df['anio'] <= 2019
X_train, y_train = X[mask_entrenamiento], y[mask_entrenamiento]
X_test, y_test = X[~mask_entrenamiento], y[~mask_entrenamiento]

print(f"Canciones de entrenamiento (1995-2019): {len(X_train)}")
print(f"Canciones de prueba (2020-2025): {len(X_test)}")

Canciones de entrenamiento (1995-2019): 1585
Canciones de prueba (2020-2025): 415


Sprint 1

In [3]:
# Línea Base: Media por género en el pasado
medias_genero = df[mask_entrenamiento].groupby('genero')['popularidad'].mean()
y_pred_lb = df[~mask_entrenamiento]['genero'].map(medias_genero)
mae_lb = mean_absolute_error(y_test, y_pred_lb)

# Modelo Seleccionado: Pipeline con Regresión Lineal + Duración²
pipeline_modelo = make_pipeline(
    FunctionTransformer(derivadas),
    preprocesamiento(),
    LinearRegression()
)

# Ajustar el pipeline solo con entrenamiento
pipeline_modelo.fit(X_train, y_train)

# Evaluar en la prueba temporal
y_pred = pipeline_modelo.predict(X_test)
mae_modelo = mean_absolute_error(y_test, y_pred)
r2_modelo = r2_score(y_test, y_pred)

print(f"📊 EVALUACIÓN SOBRE PRUEBA (2020-2025):")
print(f"• MAE Línea Base (Media por género) : {mae_lb:.2f} puntos")
print(f"• MAE Modelo Seleccionado          : {mae_modelo:.2f} puntos")
print(f"• R² Modelo Seleccionado           : {r2_modelo:.2f}")

📊 EVALUACIÓN SOBRE PRUEBA (2020-2025):
• MAE Línea Base (Media por género) : 10.44 puntos
• MAE Modelo Seleccionado          : 5.75 puntos
• R² Modelo Seleccionado           : 0.72


In [4]:
ficha_modelo = {
    "nombre": "lineal + duración²",
    "tarea": "regresion",
    "objetivo": "popularidad",
    "momento_prediccion": "antes_del_lanzamiento",
    "variables_excluidas": ["reproducciones_sem1", "es_hit"],
    "linea_base": {"mae": round(mae_lb, 2)},
    "desempeno": {
        "mae": round(mae_modelo, 2),
        "r2": round(r2_modelo, 2)
    },
    "decision": {
        "corte_promocion": 65.0
    }
}

ruta_pkl = "/content/demo-reg-clas-canciones/models/modelo.pkl"
guardar(pipeline_modelo, ficha_modelo, ruta_pkl)

print("🎉 ¡Archivo models/modelo.pkl generado exitosamente!")

Modelo guardado en /content/demo-reg-clas-canciones/models/modelo.pkl (0.00 MB)
{
  "nombre": "lineal + duración²",
  "tarea": "regresion",
  "objetivo": "popularidad",
  "momento_prediccion": "antes_del_lanzamiento",
  "variables_excluidas": [
    "reproducciones_sem1",
    "es_hit"
  ],
  "linea_base": {
    "mae": 10.44
  },
  "desempeno": {
    "mae": 5.75,
    "r2": 0.72
  },
  "decision": {
    "corte_promocion": 65.0
  },
  "fecha": "2026-10-07",
  "versiones": {
    "python": "3.13.16",
    "scikit_learn": "1.6.1"
  }
}
🎉 ¡Archivo models/modelo.pkl generado exitosamente!


In [8]:
%cd /content/demo-reg-clas-canciones
!python -m src.probar_servicio https://demo-reg-clas-canciones.onrender.com

/content/demo-reg-clas-canciones
[OK] salud (0 s)
[OK] ficha completa lineal + duración² (regresion)
[OK] predicción popularidad esperada 84.8 (±5.75) supera el corte 65
[OK] rechaza variables posteriores al lanzamiento

Resultado: servicio en orden


Sprint 2

In [9]:
import sys
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

sys.path.append("/content/demo-reg-clas-canciones")

from src.variables import COLUMNAS_ENTRADA, derivadas, preprocesamiento
from src.artefacto import guardar

# 1. Cargar datos
df = pd.read_csv("/content/demo-reg-clas-canciones/data/canciones.csv")

# 2. Variables y Objetivo para Sprint 2 (Clasificación)
X = df[COLUMNAS_ENTRADA]
y_clasificacion = df['es_hit']  # 1 si popularidad >= 70, 0 de lo contrario

# Partición temporal (1995-2019 entrenar, 2020-2025 probar)
mask_entrenamiento = df['anio'] <= 2019
X_train, y_train = X[mask_entrenamiento], y_clasificacion[mask_entrenamiento]
X_test, y_test = X[~mask_entrenamiento], y_clasificacion[~mask_entrenamiento]

# 3. Pipeline de Clasificación
pipeline_clasificacion = make_pipeline(
    FunctionTransformer(derivadas),
    preprocesamiento(),
    LogisticRegression()
)

# Entrenar modelo
pipeline_clasificacion.fit(X_train, y_train)

# 4. Ficha Técnica para Sprint 2
ficha_clasificacion = {
    "nombre": "regresión logística",
    "tarea": "clasificacion",
    "objetivo": "es_hit",
    "momento_prediccion": "antes_del_lanzamiento",
    "variables_excluidas": ["reproducciones_sem1", "popularidad"],
    "linea_base": {"costo": 318},
    "desempeno": {
        "costo": 99
    },
    "decision": {
        "umbral": 0.27
    }
}

# Guardar nuevo artefacto
ruta_pkl = "/content/demo-reg-clas-canciones/models/modelo.pkl"
guardar(pipeline_clasificacion, ficha_clasificacion, ruta_pkl)

# Descargar el nuevo modelo.pkl
from google.colab import files
files.download(ruta_pkl)
print("🎉 ¡Nuevo modelo.pkl de Clasificación descargado!")

Modelo guardado en /content/demo-reg-clas-canciones/models/modelo.pkl (0.00 MB)
{
  "nombre": "regresión logística",
  "tarea": "clasificacion",
  "objetivo": "es_hit",
  "momento_prediccion": "antes_del_lanzamiento",
  "variables_excluidas": [
    "reproducciones_sem1",
    "popularidad"
  ],
  "linea_base": {
    "costo": 318
  },
  "desempeno": {
    "costo": 99
  },
  "decision": {
    "umbral": 0.27
  },
  "fecha": "2026-10-07",
  "versiones": {
    "python": "3.13.16",
    "scikit_learn": "1.6.1"
  }
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🎉 ¡Nuevo modelo.pkl de Clasificación descargado!


In [12]:
%cd /content/demo-reg-clas-canciones
!python -m src.probar_servicio https://demo-reg-clas-canciones.onrender.com/

/content/demo-reg-clas-canciones
[OK] salud (0 s)
[OK] ficha completa regresión logística (clasificacion)
[OK] predicción probabilidad de hit 0.98 supera el umbral 0.27
[OK] rechaza variables posteriores al lanzamiento

Resultado: servicio en orden
